# 01 — Fashion-MNIST: tải và kiểm tra dataset

Notebook Julia độc lập để tải dataset chính thức, kiểm tra checksum, đọc định dạng IDX xác nhận train/test split gốc và gộp 70.000 ảnh cho EDA/FIM. Chạy cell theo thứ tự bằng Julia ≥ 1.9.

Nguồn: [Fashion-MNIST của Zalando](https://github.com/zalandoresearch/fashion-mnist), cũng được ghi trong `docs/plan.md`.

## 1. Kích hoạt môi trường của repository

Notebook tìm thư mục gốc từ thư mục đang mở, kích hoạt `Project.toml` tại đó và cài các dependencies đã khóa trong `Manifest.toml` nếu thiếu.

In [8]:
using Pkg

function find_repo_root(start=pwd())
    directory = abspath(start)
    while true
        if isfile(joinpath(directory, "Project.toml")) &&
           isfile(joinpath(directory, "docs", "plan.md")) &&
           isdir(joinpath(directory, "src"))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Không tìm thấy thư mục gốc repository.")
        directory = parent
    end
end

repo_root = find_repo_root()
Pkg.activate(repo_root)
Pkg.instantiate()

using EclatFIM
using SHA: sha256
using TOML

data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
results_dir = joinpath(repo_root, "results", "fashion")
@assert VERSION >= v"1.9"
println("Julia ", VERSION)
println("Project: ", Base.active_project())
println("Raw data: ", data_dir)


  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


Julia 1.13.3
Project: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/Project.toml
Raw data: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/data/raw/fashion-mnist


## 2. Tải bốn archive và kiểm tra SHA-256

Mỗi file được đối chiếu với checksum cố định trong source. File tải lỗi hoặc sai nội dung sẽ không được giữ làm bản dataset hợp lệ. File đã có được xác minh lại trước khi sử dụng.

In [9]:
archive_paths = download_fashion_mnist(data_dir)
expected_files = EclatFIM.FASHION_FILES

for (archive, expected) in zip(archive_paths, expected_files)
    actual = bytes2hex(sha256(read(archive)))
    @assert actual == expected.sha256
    println("SHA-256 OK  ", expected.name, "  (", filesize(archive), " bytes)")
end
println("Đã xác minh đủ 4 archive từ ", EclatFIM.FASHION_SOURCE)


SHA-256 OK  train-images-idx3-ubyte.gz  (26421880 bytes)
SHA-256 OK  train-labels-idx1-ubyte.gz  (29515 bytes)
SHA-256 OK  t10k-images-idx3-ubyte.gz  (4422102 bytes)
SHA-256 OK  t10k-labels-idx1-ubyte.gz  (5148 bytes)
Đã xác minh đủ 4 archive từ https://github.com/zalandoresearch/fashion-mnist


## 3. Đọc ảnh và nhãn IDX

IDX lưu ảnh theo thứ tự dòng. Hàm đọc chuyển đúng về mảng Julia `images[row, column, image]`, không xoay ảnh; cường độ pixel vẫn là `UInt8` trong khoảng 0–255. Nhãn lưu tách khỏi ảnh.

In [10]:
train = load_fashion_mnist(data_dir; split=:train)
test = load_fashion_mnist(data_dir; split=:test)

println("Train: ", size(train.images), " ảnh; ", length(train.labels), " nhãn")
println("Test:  ", size(test.images), " ảnh; ", length(test.labels), " nhãn")
println("Kiểu pixel: ", eltype(train.images), "; giá trị train: ", extrema(train.images))


Train: (28, 28, 60000) ảnh; 60000 nhãn
Test:  (28, 28, 10000) ảnh; 10000 nhãn
Kiểu pixel: UInt8; giá trị train: (0x00, 0xff)


## 4. Xác nhận shape, nhãn và train/test split

Các assertion xác nhận đủ 60.000 ảnh train và 10.000 ảnh test; cả hai split có ảnh 28×28, nhãn 0–9 và 6.000/1.000 ảnh mỗi lớp.

In [11]:
@assert size(train.images) == (28, 28, 60_000)
@assert size(test.images) == (28, 28, 10_000)
@assert eltype(train.images) == UInt8 && eltype(test.images) == UInt8
@assert length(train.labels) == size(train.images, 3)
@assert length(test.labels) == size(test.images, 3)
@assert all(label -> 0 <= label <= 9, train.labels)
@assert all(label -> 0 <= label <= 9, test.labels)
@assert sort(unique(train.labels)) == collect(0:9)
@assert sort(unique(test.labels)) == collect(0:9)

train_counts = [count(==(label), train.labels) for label in 0:9]
test_counts = [count(==(label), test.labels) for label in 0:9]
@assert all(==(6_000), train_counts)
@assert all(==(1_000), test_counts)

println("PASS: đúng shape, grayscale UInt8, nhãn 0–9 và split train/test.")
for label in 0:9
    println(rpad("$(label) $(FASHION_CLASSES[label + 1])", 22),
        "train=", rpad(string(train_counts[label + 1]), 7),
        "test=", test_counts[label + 1])
end


PASS: đúng shape, grayscale UInt8, nhãn 0–9 và split train/test.
0 T-shirt/top         train=6000   test=1000
1 Trouser             train=6000   test=1000
2 Pullover            train=6000   test=1000
3 Dress               train=6000   test=1000
4 Coat                train=6000   test=1000
5 Sandal              train=6000   test=1000
6 Shirt               train=6000   test=1000
7 Sneaker             train=6000   test=1000
8 Bag                 train=6000   test=1000
9 Ankle boot          train=6000   test=1000


## 5. Gộp train và test cho EDA/FIM

EDA-1 khảo sát toàn bộ **70.000 ảnh**, gồm 60.000 ảnh train và 10.000 ảnh test gốc. Cell này nối theo trục ảnh thứ ba, giữ thứ tự train trước, test sau; không trộn ngẫu nhiên hay xóa ảnh trùng.

`dataset.images` và `dataset.labels` là đầu vào chung cho EDA-1. `source_split` và `source_index` cho phép truy ngược mỗi ảnh về split và chỉ số ban đầu. Split gốc chỉ phục vụ truy vết trong phạm vi EDA này.

Bộ gộp được lưu thành **70.000 file ảnh PNG grayscale 28×28** tại `data/processed/fashion-mnist/combined/images/`, tên `000001.png` đến `070000.png`. Nhãn, split/index nguồn, đường dẫn và SHA-256 nằm trong `image_index.csv`; thông tin tổng hợp nằm ở `metadata.toml`. Cell đọc lại và đối chiếu mọi pixel của từng PNG; chạy lại sẽ ghi lại cùng các tên file.


In [12]:
n_train = size(train.images, 3)
n_test = size(test.images, 3)

dataset = (
    images=cat(train.images, test.images; dims=3),
    labels=vcat(train.labels, test.labels),
    split=:combined,
    source_split=vcat(fill(:train, n_train), fill(:test, n_test)),
    source_index=vcat(collect(1:n_train), collect(1:n_test)),
)

# Tên ngắn dùng trong plan EDA-1: X[row, column, image_id], y[image_id].
X = dataset.images
y = dataset.labels
combined_counts = [count(==(label), y) for label in 0:9]

@assert size(X) == (28, 28, 70_000)
@assert eltype(X) == UInt8
@assert length(y) == length(dataset.source_split) == length(dataset.source_index) == 70_000
@assert all(==(7_000), combined_counts)
@assert view(X, :, :, 1:n_train) == train.images
@assert view(X, :, :, n_train + 1:n_train + n_test) == test.images
@assert y[1:n_train] == train.labels && y[n_train + 1:end] == test.labels
@assert dataset.source_split[1] == :train && dataset.source_split[end] == :test
@assert dataset.source_index[n_train] == n_train && dataset.source_index[n_train + 1] == 1

println("Combined dataset: ", size(X), "; labels: ", length(y))
println("Mỗi lớp: ", combined_counts)
println("Thứ tự image_id: 1:60_000 từ train; 60_001:70_000 từ test.")

# Xuất mỗi ảnh thành PNG grayscale 8-bit, giữ nguyên 28×28 pixel.
using PNGFiles
using ColorTypes: Gray
using FixedPointNumbers: N0f8

combined_dir = joinpath(repo_root, "data", "processed", "fashion-mnist", "combined")
images_dir = joinpath(combined_dir, "images")
mkpath(images_dir)
index_path = joinpath(combined_dir, "image_index.csv")

function export_combined_pngs(images_dir, index_path, dataset)
    temporary_index = index_path * ".tmp"
    open(temporary_index, "w") do io
        println(io, "image_id,relative_path,label,source_split,source_index,sha256")
        for image_id in eachindex(dataset.labels)
            filename = lpad(string(image_id), 6, '0') * ".png"
            path = joinpath(images_dir, filename)
            image = @view dataset.images[:, :, image_id]
            # reinterpret giữ đúng 8 bit gốc; không chuẩn hóa qua số thực hay resize.
            PNGFiles.save(path, Gray.(reinterpret(N0f8, image)))

            # Kiểm tra chính file vừa ghi, gồm shape, kiểu grayscale và mọi pixel.
            decoded = PNGFiles.load(path)
            @assert size(decoded) == (28, 28)
            @assert eltype(decoded) == Gray{N0f8}
            @assert reinterpret(UInt8, decoded) == image
            checksum = bytes2hex(open(sha256, path))
            println(io, "$image_id,images/$filename,$(dataset.labels[image_id]),$(dataset.source_split[image_id]),$(dataset.source_index[image_id]),$checksum")
            if image_id % 10_000 == 0
                println("Đã lưu và kiểm tra ", image_id, " / ", length(dataset.labels), " ảnh PNG")
            end
        end
    end
    mv(temporary_index, index_path; force=true)
end

export_combined_pngs(images_dir, index_path, dataset)
@assert count(endswith(".png"), readdir(images_dir)) == 70_000
index_rows = readlines(index_path)
@assert length(index_rows) == 70_001
for image_id in eachindex(y)
    fields = split(index_rows[image_id + 1], ',')
    @assert parse(Int, fields[1]) == image_id
    @assert fields[2] == "images/" * lpad(string(image_id), 6, '0') * ".png"
    @assert parse(Int, fields[3]) == y[image_id]
    @assert fields[4] == string(dataset.source_split[image_id])
    @assert parse(Int, fields[5]) == dataset.source_index[image_id]
end

combined_metadata = Dict(
    "dataset" => "Fashion-MNIST combined",
    "source" => EclatFIM.FASHION_SOURCE,
    "format" => "individual PNG files, grayscale 8-bit",
    "images" => length(y),
    "image_shape" => [28, 28],
    "pixel_type" => "UInt8",
    "class_counts" => combined_counts,
    "original_train_images" => n_train,
    "original_test_images" => n_test,
    "order" => "train first, then test; original order retained",
    "index_base" => 1,
    "roundtrip_verified" => true,
    "verified_png_images" => length(y),
    "sha256" => Dict("image_index.csv" => bytes2hex(open(sha256, index_path))),
    "png_checksums" => "sha256 column in image_index.csv",
    "source_archives" => [Dict("filename" => file.name, "sha256" => file.sha256)
        for file in expected_files],
)
metadata_path = joinpath(combined_dir, "metadata.toml")
open(metadata_path * ".tmp", "w") do io
    TOML.print(io, combined_metadata; sorted=true)
end
mv(metadata_path * ".tmp", metadata_path; force=true)

# Thay bản combined IDX gzip đã xuất ở phiên bản cell trước bằng folder PNG.
# Chỉ bỏ đúng hai file generated cũ sau khi cả 70.000 PNG đã được xác minh.
for filename in ("images-idx3-ubyte.gz", "labels-idx1-ubyte.gz")
    old_path = joinpath(combined_dir, filename)
    isfile(old_path) && rm(old_path)
end
println("Dataset PNG: ", relpath(combined_dir, repo_root))
println("Nhãn, nguồn ảnh và checksum: ", relpath(index_path, repo_root))


Combined dataset: (28, 28, 70000); labels: 70000
Mỗi lớp: [7000, 7000, 7000, 7000, 7000, 7000, 7000, 7000, 7000, 7000]
Thứ tự image_id: 1:60_000 từ train; 60_001:70_000 từ test.


[ Info: Precompiling PNGFiles [f57f5aa1-a3ce-4bc8-8ab9-96f992907883] (caches not reused: 1 for different dependency version already loaded)
Precompiling packages...
  15.9 s  ✓ ImageCore


Đã lưu và kiểm tra 10000

   4.1 s  ✓ PNGFiles
  2 dependencies successfully precompiled in 20 seconds. 26 already precompiled.
  1 dependency precompiled but a different version is currently loaded (PrecompileTools). Restart julia to access the new version. Otherwise, 2 dependents of this package may trigger further precompilation to work with the unexpected version.


 / 70000 ảnh PNG
Đã lưu và kiểm tra 20000 / 70000 ảnh PNG
Đã lưu và kiểm tra 30000 / 70000 ảnh PNG
Đã lưu và kiểm tra 40000 / 70000 ảnh PNG
Đã lưu và kiểm tra 50000 / 70000 ảnh PNG
Đã lưu và kiểm tra 60000 / 70000 ảnh PNG
Đã lưu và kiểm tra 70000 / 70000 ảnh PNG
Dataset PNG: data/processed/fashion-mnist/combined
Nhãn, nguồn ảnh và checksum: data/processed/fashion-mnist/combined/image_index.csv


## 6. Lưu báo cáo xác minh

Báo cáo ghi nguồn, phiên bản Julia, số ảnh theo split gốc, bộ gộp 70.000 ảnh và checksum. Các mảng ảnh không được ghi vào notebook nên file vẫn gọn.

In [13]:
mkpath(results_dir)
report = Dict(
    "dataset" => "Fashion-MNIST",
    "source" => EclatFIM.FASHION_SOURCE,
    "julia_version" => string(VERSION),
    "image_shape" => [28, 28],
    "pixel_type" => string(eltype(train.images)),
    "checks_passed" => true,
    "eda_scope" => "combined_train_test",
    "combined_images" => size(X, 3),
    "combined_folder" => relpath(combined_dir, repo_root),
    "combined_format" => "individual PNG grayscale 8-bit",
    "combined_index" => relpath(index_path, repo_root),
    "combined_roundtrip_verified" => true,
    "combined_class_counts" => combined_counts,
    "combined_pixel_count" => length(X),
    "combined_order" => "train first, then test; original order retained",
    "provenance" => "image_index.csv: image_id, relative_path, label, source_split, source_index, sha256",
    "train_images" => size(train.images, 3),
    "test_images" => size(test.images, 3),
    "train_class_counts" => train_counts,
    "test_class_counts" => test_counts,
    "archives" => [Dict("filename" => file.name, "sha256" => file.sha256)
        for file in expected_files],
)
report_path = joinpath(results_dir, "dataset_validation.toml")
open(report_path, "w") do io
    TOML.print(io, report; sorted=true)
end
println("Báo cáo: ", relpath(report_path, repo_root))


Báo cáo: results/fashion/dataset_validation.toml


Dataset gộp đã được lưu dưới dạng file ảnh:

```text
data/processed/fashion-mnist/combined/
├── images/000001.png ... 070000.png
├── image_index.csv
└── metadata.toml
```

Mỗi dòng CSV tương ứng một ảnh, gồm `image_id,relative_path,label,source_split,source_index,sha256`. Đọc CSV theo `image_id` để giữ đúng nhãn và thứ tự, không dựa vào thứ tự liệt kê file của hệ điều hành.

Ví dụ đọc một ảnh đã lưu bằng Julia:

```julia
using PNGFiles
image = PNGFiles.load(joinpath(combined_dir, "images", "000001.png"))
pixels = reinterpret(UInt8, image)  # 28×28, đúng cường độ 0–255 gốc
```

`metadata.toml` lưu số lượng, class counts, nguồn và checksum của CSV. Checksum từng PNG nằm trong CSV. Toàn bộ folder dataset được Git ignore.
